In [2]:
import numpy as np

### Speicfy setup of qubits

In [3]:
rng = np.random.default_rng(1234)

In [6]:
# Basic utilities
def normalise(psi):
    norm = np.linalg.norm(psi)
    if norm == 0:
        return psi
    return psi / norm

def dagger(psi):
    return np.conjugate(psi.T)

def density_matrix(psi):
    psi = normalise(psi)
    return np.outer(psi, dagger(psi))

def tensor_product(A,B):
    return np.kron(A,B)

state_0 = np.array([1,0])
state_1 = np.array([0,1])


#### Trial 2-qubit state

In [7]:
# Use bell state as a trial

state_bell = normalise(tensor_product(state_0, state_0) + tensor_product(state_1, state_1))
rho_bell = density_matrix(state_bell)

In [9]:
print(rho_bell)
print(np.trace(rho_bell))  # Should be 1

[[0.5 0.  0.  0.5]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.  0.5]]
1.0000000000000002


Define POVM elements to measure in Z basis

In [14]:
Z = [[1,0],[0,-1]]
ZZ = tensor_product(Z,Z)
print(ZZ)

[[ 1  0  0  0]
 [ 0 -1  0  0]
 [ 0  0 -1  0]
 [ 0  0  0  1]]


Can see from the ZZ matrix that the eigenvalues are +1 (for |00> and |11>) and -1 (for |10> and |01>)

In tomography, you aim to reconstruct the density matrix. Using ZZ gives an eigenvalue [-1,1] simply saying how likely the qubits are to be aligned or misaligned. Projectors (such as |00><00|) give individual outcomes of the ZZ measurement (the entire joint probability distribution).

Projectors and density matrices are visually and mathematically identical, but projectors are used to measure states and density matrices represent prepared states

The POVM is a list of the projectors defined by the eigenstates of the operator, ZZ in this case. This POVM is related to ZZ as we are using its eigenstates as the projectors.

In [15]:
E_00 = density_matrix(tensor_product(state_0, state_0))  # |00><00|
E_01 = density_matrix(tensor_product(state_0, state_1))  # |01><01|
E_10 = density_matrix(tensor_product(state_1, state_0))  # |10><10|
E_11 = density_matrix(tensor_product(state_1, state_1))  # |11><11|

POVM_ZZ = [E_00, E_01, E_10, E_11]

In [21]:
probs = [float(np.trace(rho_bell @ E)) for E in POVM_ZZ]
# Finds probabilities of measuring each outcome in the POVM given the state rho_bell

In [22]:
print(probs)

[0.5000000000000001, 0.0, 0.0, 0.5000000000000001]


Probabilities have come out as expected for Bell State

### Constructing POVMs using X, Y, and Z basis

In [23]:
# The three Pauli matrices

X = np.array([[0, 1],
              [1, 0]], dtype=complex)

Y = np.array([[0, -1j],
              [1j, 0]], dtype=complex)

Z = np.array([[1, 0],
              [0, -1]], dtype=complex)

In [63]:
# States for X basis
states_X = np.linalg.eig(X)[1]
states_Y = np.linalg.eig(Y)[1]
states_Z = np.linalg.eig(Z)[1]

In [78]:
# Define a function to get POVM from two sets of basis states
def get_POVM(A,B):
    POVM = []
    E_0 = density_matrix(tensor_product(A[0], B[0]))
    E_1 = density_matrix(tensor_product(A[0], B[1]))
    E_2 = density_matrix(tensor_product(A[1], B[0]))
    E_3 = density_matrix(tensor_product(A[1], B[1]))
    POVM = [E_0, E_1, E_2, E_3]
    return POVM
# Note for future that density_matrix automatically normalises the states

In [79]:
POVM_XX = get_POVM(states_X, states_X)
POVM_XY = get_POVM(states_X, states_Y)
POVM_XZ = get_POVM(states_X, states_Z)
POVM_YX = get_POVM(states_Y, states_X)
POVM_YY = get_POVM(states_Y, states_Y)
POVM_YZ = get_POVM(states_Y, states_Z)
POVM_ZX = get_POVM(states_Z, states_X)
POVM_ZY = get_POVM(states_Z, states_Y)
POVM_ZZ = get_POVM(states_Z, states_Z)

In [87]:
# Check that POVMs sum to identity
print(sum(POVM_XX).round(6))

[[ 1.+0.j  0.+0.j -0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j  0.+0.j  0.+0.j]
 [-0.+0.j  0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  1.+0.j]]


In [88]:
def get_probs(rho, POVM):
    return [float(np.trace(rho @ E)) for E in POVM]

In [90]:
probs_XX = get_probs(rho_bell, POVM_XX)
probs_XY = get_probs(rho_bell, POVM_XY)
probs_XZ = get_probs(rho_bell, POVM_XZ)
probs_YX = get_probs(rho_bell, POVM_YX)
probs_YY = get_probs(rho_bell, POVM_YY)
probs_YZ = get_probs(rho_bell, POVM_YZ)
probs_ZX = get_probs(rho_bell, POVM_ZX)
probs_ZY = get_probs(rho_bell, POVM_ZY)
probs_ZZ = get_probs(rho_bell, POVM_ZZ)

/var/folders/fx/lnm0r8px7dl3hm8pdd5ryl4w0000gn/T/ipykernel_9563/1542495328.py:2: ComplexWarning: Casting complex values to real discards the imaginary part
  return [float(np.trace(rho @ E)) for E in POVM]


**We constructed all nine two-qubit product-Pauli POVMs (XX, XY, …, ZZ) and used them to compute, via Born’s rule, the exact theoretical measurement probabilities for each outcome of the Bell state.**

### Generating noisy bell state shots - aim to reconstruct Bell State density matrix from these

In [93]:
# Generate with gaussian noise to start with

def generate_noisy_shot(probs, noise_level):
    noisy_probs = probs + rng.normal(0, noise_level, size=len(probs))
    noisy_probs = np.clip(noisy_probs, 0, None)  # Ensure no negative probabilities
    noisy_probs /= np.sum(noisy_probs)  # Renormalise
    shot = rng.multinomial(1, noisy_probs) # Draw probabilistic shots
    return shot
# Noise is remade for each shot to simulate noise in a real experiment

In [99]:
# trial for ZZ POVM
N_shots = 1000
noise_level = 0.05  # Standard deviation of Gaussian noise
shots_ZZ = [generate_noisy_shot(probs_ZZ, noise_level) for _ in range(N_shots)]

In [102]:
measured_probs_ZZ = np.mean(shots_ZZ, axis=0)
print("Theoretical probs ZZ:", probs_ZZ)
print("Measured probs ZZ:", measured_probs_ZZ)

Theoretical probs ZZ: [0.5000000000000001, 0.0, 0.0, 0.5000000000000001]
Measured probs ZZ: [0.468 0.021 0.018 0.493]


In [103]:
shots_XX = [generate_noisy_shot(probs_XX, noise_level) for _ in range(N_shots)]
shots_XY = [generate_noisy_shot(probs_XY, noise_level) for _ in range(N_shots)]
shots_XZ = [generate_noisy_shot(probs_XZ, noise_level) for _ in range(N_shots)]
shots_YX = [generate_noisy_shot(probs_YX, noise_level) for _ in range(N_shots)]
shots_YY = [generate_noisy_shot(probs_YY, noise_level) for _ in range(N_shots)]
shots_YZ = [generate_noisy_shot(probs_YZ, noise_level) for _ in range(N_shots)]
shots_ZX = [generate_noisy_shot(probs_ZX, noise_level) for _ in range(N_shots)]
shots_ZY = [generate_noisy_shot(probs_ZY, noise_level) for _ in range(N_shots)]

In [104]:
measured_probs_XX = np.mean(shots_XX, axis=0)
measured_probs_XY = np.mean(shots_XY, axis=0)
measured_probs_XZ = np.mean(shots_XZ, axis=0)
measured_probs_YX = np.mean(shots_YX, axis=0)
measured_probs_YY = np.mean(shots_YY, axis=0)
measured_probs_YZ = np.mean(shots_YZ, axis=0)
measured_probs_ZX = np.mean(shots_ZX, axis=0)
measured_probs_ZY = np.mean(shots_ZY, axis=0)

### Putting things in place to reconstruct density matrix